In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts.datasets import ILINetDataset
import torch

# fix the random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)


# --- load dataset (instantiate!) ---
ili = ILINetDataset().load()

# --- pick a "weighted" component robustly ---
comps = list(getattr(ili, "components", []))
if len(comps) == 0:
    # already univariate
    ili_w = ili
    print("ILINet appears univariate.")
else:
    # prefer something containing "WEIGHTED"
    weighted = None
    for c in comps:
        if "WEIGHTED" in str(c).upper():
            weighted = c
            break
    if weighted is None:
        # fallback: first component
        weighted = comps[0]
        print("No obvious 'WEIGHTED' component found; using first component:", weighted)
    else:
        print("Using component:", weighted)
    ili_w = ili[weighted]

print("Span:", ili_w.start_time(), "->", ili_w.end_time(), "| freq:", ili_w.freq, "| len:", len(ili_w))

In [ ]:
import pandas as pd
from darts import TimeSeries
from darts import TimeSeries

N = len(ili_w)
n_train = int(np.floor(0.7 * N))
n_val   = int(np.floor(0.1 * N))
n_test  = N - n_train - n_val

train_raw = ili_w[:n_train]
val_raw   = ili_w[n_train:n_train + n_val]
test_raw  = ili_w[n_train + n_val:]

print("Split lengths:", len(train_raw), len(val_raw), len(test_raw))
print("Train end:", train_raw.end_time())
print("Val span:", val_raw.start_time(), "->", val_raw.end_time())
print("Test span:", test_raw.start_time(), "->", test_raw.end_time())
def fill_missing(ts):
    # convert to pandas, interpolate, then back to TimeSeries
    if hasattr(ts, "pd_dataframe"):
        df = ts.pd_dataframe()
    else:
        df = ts.to_dataframe()
    df = df.interpolate(limit_direction="both")
    df = df.ffill().bfill()
    return TimeSeries.from_dataframe(df)

train_raw = fill_missing(train_raw)
val_raw   = fill_missing(val_raw)
test_raw  = fill_missing(test_raw)

In [ ]:

from darts import TimeSeries
def ts_values(ts):
    return ts.values(copy=False).squeeze(-1).astype(np.float64)

def zscore(ts, mu, sd):
    v = (ts_values(ts) - mu) / sd
    return TimeSeries.from_times_and_values(ts.time_index, v.astype(np.float32))

mu = float(np.nanmean(ts_values(train_raw)))
sd = float(np.nanstd(ts_values(train_raw), ddof=0))
if (not np.isfinite(sd)) or sd < 1e-8:
    sd = 1.0

train = zscore(train_raw, mu, sd)
val   = zscore(val_raw, mu, sd)
test  = zscore(test_raw, mu, sd)

print("mu, sd:", mu, sd)

In [ ]:
import torch
from pytorch_lightning.callbacks import EarlyStopping
from darts.models import TCNModel

IN_LEN = 26
if len(train) <= IN_LEN + 5:
    IN_LEN = max(12, len(train) // 4)
    print("Train too short; using IN_LEN =", IN_LEN)

OUT_LEN = 1

early_stopper = EarlyStopping("val_loss", min_delta=1e-3, patience=3, verbose=True)

if torch.cuda.is_available():
    pl_trainer_kwargs = {"accelerator":"gpu","devices":1,"num_sanity_val_steps":0,"gradient_clip_val":0.1,"callbacks":[early_stopper]}
    num_workers = 4
elif torch.backends.mps.is_available():
    pl_trainer_kwargs = {"accelerator":"mps","devices":1,"num_sanity_val_steps":0,"gradient_clip_val":0.1,"callbacks":[early_stopper]}
    num_workers = 0
else:
    pl_trainer_kwargs = {"accelerator":"cpu","devices":1,"num_sanity_val_steps":0,"gradient_clip_val":0.1,"callbacks":[early_stopper]}
    num_workers = 0

tcn = TCNModel(
    input_chunk_length=IN_LEN,
    output_chunk_length=OUT_LEN,
    batch_size=32,
    n_epochs=200,
    kernel_size=5,
    num_filters=8,
    dilation_base=2,
    dropout=0.2,
    optimizer_kwargs={"lr": 1e-3},
    pl_trainer_kwargs=pl_trainer_kwargs,
    model_name="tcn_ilinet_weighted_5_1_4",
    force_reset=True,
    save_checkpoints=True,
)

# longer val: include IN_LEN history before val window
train_val = train.concatenate(val)
val_for_training = train_val[-(IN_LEN + len(val)):]

tcn.fit(
    series=train,
    val_series=val_for_training,
    max_samples_per_ts=2000,
    dataloader_kwargs={"num_workers": num_workers},
    verbose=True,
)

tcn = TCNModel.load_from_checkpoint("tcn_ilinet_weighted_5_1_4", best=True)
print("Loaded best checkpoint.")

In [ ]:
from darts.timeseries import concatenate
from darts.metrics import mae

def _maybe_concat(x):
    return concatenate(x, axis=0) if isinstance(x, list) else x

# context = train + val
ctx = train.concatenate(val)
full = ctx.concatenate(test)

preds = tcn.historical_forecasts(
    series=full,
    start=test.start_time(),
    forecast_horizon=1,
    stride=1,
    retrain=False,
    last_points_only=True,
    verbose=False,
)
preds = _maybe_concat(preds).slice_intersect(test)

yy = test.slice_intersect(preds)
pp = preds.slice_intersect(yy)
mae_test = mae(yy, pp)
print("Test 1-step MAE (z):", float(mae_test))

In [ ]:
import time
import numpy as np
import pandas as pd
from darts.timeseries import concatenate
from tqdm.auto import tqdm
def iter_data(data, show_pbar=True, desc=""):
    return tqdm(data, desc=desc, leave=False) if show_pbar else data
# ---------- helpers ----------
def _maybe_concat(x):
    return concatenate(x, axis=0) if isinstance(x, list) else x

def build_X_t_rawwindow_weekly(full_series, pred_ts, L):
    """
    High-dim raw feature = past L target values (L dims).
    """
    v = full_series.values(copy=False).squeeze(-1).astype(np.float32)
    time_index = full_series.time_index
    idx_map = {t: i for i, t in enumerate(time_index)}

    X = np.zeros((len(pred_ts), L), dtype=np.float32)
    for j, ts in enumerate(pred_ts):
        i = idx_map[ts]
        start = i - L
        end = i
        if start < 0:
            pad = -start
            yw = np.concatenate([np.full((pad,), v[0], np.float32), v[:end]], axis=0)[-L:]
        else:
            yw = v[start:end]
        X[j, :] = yw
    return X

def precompute_tcn_valtest_data_ili(
    tcn,
    train, val, test,         # each is a univariate TimeSeries (weekly)
    IN_LEN=52,
    constant_eps=1e-8,
):
    """
    We compute 1-step forecasts over val+test via historical_forecasts (teacher forced).
    """
    tv = train.concatenate(val)
    full = tv.concatenate(test)

    preds = _maybe_concat(tcn.historical_forecasts(
        series=full,
        start=val.start_time(),
        forecast_horizon=1,
        stride=1,
        retrain=False,
        last_points_only=True,
        verbose=False,
    ))

    y = full.slice_intersect(preds)
    preds = preds.slice_intersect(y)

    yv = y.values(copy=False).squeeze(-1).astype(np.float32)
    pv = preds.values(copy=False).squeeze(-1).astype(np.float32)
    sv = np.abs(yv - pv).astype(np.float32)

    ts = y.time_index
    is_test = np.array([t >= test.start_time() for t in ts], dtype=bool)

    y_test = yv[is_test]
    const_test = (np.nanmax(y_test) - np.nanmin(y_test)) < constant_eps if y_test.size else True

    X = build_X_t_rawwindow_weekly(full_series=full, pred_ts=ts, L=IN_LEN)

    return [{
        "series_id": 0,
        "ts": ts,
        "y": yv,
        "pred": pv,
        "score": sv,
        "X": X,
        "is_test": is_test,
        "const_test": const_test,
    }]

# ---- choose ILI-appropriate windows (weeks) ----
ALPHA = 0.1
IN_LEN = 26          # feature window (weeks)
R = 52               # calibration window (weeks)

data = precompute_tcn_valtest_data_ili(tcn=tcn, train=train, val=val, test=test, IN_LEN=IN_LEN)
print("Built data cache for", len(data), "series (ILI should be 1).")
print("Constant test:", data[0]["const_test"])
print("X dim:", data[0]["X"].shape[1])
print("Total points (val+test):", len(data[0]["y"]), "| test points:", int(data[0]["is_test"].sum()))

In [ ]:
from cp_methods import eval_cp, eval_lcp, eval_aci, eval_dtaci, eval_olcp, eval_olcp_adahedge, eval_spci
METHOD_RUNNERS = {
    "cp": eval_cp,
    "lcp": eval_lcp,
    "aci": eval_aci,
    "dtaci": eval_dtaci,
    "olcp": eval_olcp,
    "olcph": eval_olcp_adahedge,
    "spci": lambda data, alpha, R, exclude_const_test=False: eval_spci(
        data,
        alpha=alpha,
        R=R,
        w_lag=8,          # weeks 
        T_train=R,        # comparable
        refit_every=1,    # refit monthly-ish (every 4 weeks); 1 is most faithful
        exclude_const_test=exclude_const_test,
        show_pbar=True,
    ),
}


def summarize_by_series(df, name):
    """
    For each series_id:
      - coverage_i = mean(covered) over that series' rows
      - width_i    = mean(width) over that series' finite rows
    Then report mean/std across series.
    """
    d = df.copy()
    d["width"] = d["width"].replace([np.inf, -np.inf], np.nan)

    g = d.groupby("series_id", observed=True)
    per = g.agg(
        cov=("covered", "mean"),
        width=("width", "mean"),
        n=("covered", "size"),
        n_finite_width=("width", lambda x: int(np.isfinite(x).sum())),
    ).reset_index()

    # mean/std across series
    cov_mean = float(per["cov"].mean())
    wid_mean = float(per["width"].mean())

    return {
        "Method": name,
        "Cov_mean": cov_mean,
        "Width_mean": wid_mean,
        "Rows": int(len(d)),
    }, per



results = {}
summary_rows = []
per_series_tables = {}

for name, fn in METHOD_RUNNERS.items():
    df_m, sec = fn(data, alpha=ALPHA, R=R, exclude_const_test=False)
    results[name] = df_m

    row, per = summarize_by_series(df_m, name.upper())
    row["Seconds"] = float(sec)
    summary_rows.append(row)
    per_series_tables[name] = per

summary = pd.DataFrame(summary_rows).sort_values("Seconds")

print(summary.to_string(index=False, formatters={
    "Cov_mean": "{:.3f}".format,
    "Width_mean": "{:.4f}".format,
    "Seconds": "{:.2f}".format,
}))

In [ ]:
import numpy as np
import pandas as pd

def _mbb_indices(n, block_len, rng):
    """
    Moving block bootstrap indices for length n.
    Samples ceil(n/block_len) blocks of consecutive indices.
    """
    block_len = int(max(1, min(block_len, n)))
    n_blocks = int(np.ceil(n / block_len))
    # valid block starts
    starts = rng.integers(0, n - block_len + 1, size=n_blocks)
    idx = np.concatenate([np.arange(s, s + block_len) for s in starts], axis=0)
    return idx[:n]

def block_bootstrap_se(df, block_len=13, n_boot=2000, seed=0):
    """
    df must have columns: 'covered' (bool) and 'width' (float).
    Returns mean, SE, and 95% CI for coverage and width using MBB.
    """
    d = df.sort_values("t").copy()
    covered = d["covered"].to_numpy(dtype=float)  # 0/1
    width = d["width"].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)

    n = len(d)
    rng = np.random.default_rng(seed)

    cov_boot = np.empty(n_boot, dtype=float)
    wid_boot = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        idx = _mbb_indices(n, block_len, rng)
        cov_boot[b] = np.mean(covered[idx])
        wid_boot[b] = np.nanmean(width[idx])

    out = {
        "cov_mean": float(np.mean(covered)),
        "cov_se": float(np.std(cov_boot, ddof=1)),
        "cov_ci_lo": float(np.quantile(cov_boot, 0.025)),
        "cov_ci_hi": float(np.quantile(cov_boot, 0.975)),
        "wid_mean": float(np.nanmean(width)),
        "wid_se": float(np.std(wid_boot, ddof=1)),
        "wid_ci_lo": float(np.quantile(wid_boot, 0.025)),
        "wid_ci_hi": float(np.quantile(wid_boot, 0.975)),
        "n": int(n),
        "block_len": int(min(block_len, n)),
        "n_boot": int(n_boot),
    }
    return out

In [ ]:
def bootstrap_summary_table(results, block_len=13, n_boot=2000, seed=0):
    rows = []
    for name, df in results.items():
        stats = block_bootstrap_se(df, block_len=block_len, n_boot=n_boot, seed=seed)
        rows.append({
            "Method": name.upper(),
            "Coverage": stats["cov_mean"],
            "Cov_SE": stats["cov_se"],
            # "Cov_CI": f"[{stats['cov_ci_lo']:.3f}, {stats['cov_ci_hi']:.3f}]",
            "Width": stats["wid_mean"],
            "Wid_SE": stats["wid_se"],
            # "Wid_CI": f"[{stats['wid_ci_lo']:.3f}, {stats['wid_ci_hi']:.3f}]",
            "n": stats["n"],
            "B": stats["block_len"],
        })
    return pd.DataFrame(rows).sort_values("Method")

boot_tab = bootstrap_summary_table(results, block_len=26, n_boot=1000, seed=42)
print(boot_tab.to_string(index=False, formatters={
    "Coverage": "{:.3f}".format,
    "Cov_SE": "{:.3f}".format,
    "Width": "{:.4f}".format,
    "Wid_SE": "{:.4f}".format,
}))